In [10]:
import pandas as pd
import numpy as np
import sys, os
import matplotlib.pyplot as plt
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

In [11]:
df_train_raw = pd.read_csv('../data/raw/train.csv').drop('Id', axis=1)
df_train = df_train_raw.copy()
print(f'start dimentions --> {df_train_raw.shape}')
df_train_raw.head()

start dimentions --> (1460, 80)


,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,FR2,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,Inside,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,Corner,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,FR2,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [12]:
#Analisis de datos faltantes
missing_data = df_train_raw.isnull().sum()
missing_data = pd.DataFrame(missing_data[missing_data > 0], columns=['missing_count'])
missing_data['missing_percentage (%)'] = np.round(missing_data['missing_count'] / df_train_raw.shape[0] * 100, 2)
missing_data = missing_data.sort_values(by='missing_count', ascending=False)
print('fields with missing data:')
missing_data

fields with missing data:


,missing_count,missing_percentage (%)
PoolQC,1453,99.52
MiscFeature,1406,96.30
Alley,1369,93.77
Fence,1179,80.75
MasVnrType,872,59.73
FireplaceQu,690,47.26
LotFrontage,259,17.74
GarageType,81,5.55
GarageYrBlt,81,5.55
GarageFinish,81,5.55


In [13]:
#Manejo de datos faltantes
#Paso inicial eliminar las columnas que tienen mas del 80% de datos faltantes

#Alley: El tipo de acceso al callejón.
#MasVnrType: El tipo de revestimiento de mampostería.
#FireplaceQu: La calidad de la chimenea.
#PoolQC: La calidad de la piscina.
#Fence: La calidappd de la valla.
#MiscFeature: Una característica miscelánea no cubierta en otras categorías.

print(f'start dimentions {df_train.shape}')
initial_columns = df_train.columns
df_train = df_train.dropna(thresh=len(df_train) * 0.8, axis=1)
columns_deleted = [col for col in initial_columns if col not in df_train.columns]
print(f'columns deleted ({len(columns_deleted)}) --> {columns_deleted}')

start dimentions (1460, 80)
columns deleted (6) --> ['Alley', 'MasVnrType', 'FireplaceQu', 'PoolQC', 'Fence', 'MiscFeature']


In [14]:
#Columnas con datos faltantes resultantes de la eliminacion anterior
missing_res_deleted = df_train.isnull().sum()
missing_res_deleted = missing_res_deleted[missing_res_deleted > 0].sort_values(ascending=True)
print(f'numero de columnas con datos faltantes --> {len(missing_res_deleted)}')
missing_res_deleted

numero de columnas con datos faltantes --> 13


Electrical        1
MasVnrArea        8
BsmtCond         37
BsmtQual         37
BsmtFinType1     37
BsmtExposure     38
BsmtFinType2     38
GarageType       81
GarageCond       81
GarageYrBlt      81
GarageFinish     81
GarageQual       81
LotFrontage     259
dtype: int64

In [15]:
#imputacion simple
#Electrical: El sistema eléctrico.
df_train['Electrical'] = df_train['Electrical'].fillna(df_train['Electrical'].mode()[0])
#MasVnrArea: Área de revestimiento de mampostería en pies cuadrados.
df_train['MasVnrArea'] = df_train['MasVnrArea'].fillna(0)
#TotalBsmtSF: Metros cuadrados totales del sótano. --> (0 = no tiene sótano)
#BsmtUnfSF: Metros cuadrados del sótano sin terminar. --> (0 != tiene sótano || 0 != tiene sótano sin terminar)
#BsmtQual: Evalúa la altura del sótano.
#BsmtCond: Evalúa la condición general del sótano.
#GarageType: La ubicación del garaje. --> ( Basment VALOR DE INTERES)
#BsmtExposure: Refleja la cantidad de exposición al sótano al aire exterior.
#BsmtFinType1: Evaluación del tipo de acabado del sótano.
#BsmtFinType2: Evaluación del tipo de acabado del sótano (si hay dos tipos).
#BsmtFullBath: Número de baños completos en el sótano. --> (Diferente de 0 VALOR DE INTERES)
#BsmtHalfBath: Número de medios baños en el sótano. --> (Diferente de 0 VALOR DE INTERES)
#FireplaceQu: La calidad de la chimenea. (DELETED)
""" bsmt = df_train[['TotalBsmtSF', 'BsmtUnfSF', 'BsmtQual', 'GarageType', 'BsmtFullBath', 'BsmtHalfBath', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2']] """
""" bsmt = bsmt[bsmt['BsmtQual'].isnull() | bsmt['BsmtCond'].isnull() | bsmt['BsmtExposure'].isnull() | bsmt['BsmtFinType1'].isnull() | bsmt['BsmtFinType2'].isnull() |
            bsmt['BsmtUnfSF'].isnull() | bsmt['TotalBsmtSF'].isnull() | bsmt['BsmtFullBath'].isnull() | bsmt['BsmtHalfBath'].isnull() | (bsmt['GarageType'] == 'Basment')] """

" bsmt = bsmt[bsmt['BsmtQual'].isnull() | bsmt['BsmtCond'].isnull() | bsmt['BsmtExposure'].isnull() | bsmt['BsmtFinType1'].isnull() | bsmt['BsmtFinType2'].isnull() |\n            bsmt['BsmtUnfSF'].isnull() | bsmt['TotalBsmtSF'].isnull() | bsmt['BsmtFullBath'].isnull() | bsmt['BsmtHalfBath'].isnull() | (bsmt['GarageType'] == 'Basment')] "

In [16]:
cond_not_basement = (df_train['TotalBsmtSF'] == 0) & (df_train['BsmtUnfSF'] == 0)
#BsmtCond: Evalúa la condición general del sótano. -> (NA) no tiene sótano
#BsmtQual: Evalúa la altura del sótano. -> (NA) no tiene sótano
##BsmtFinType1: Evaluación del tipo de acabado del sótano. -> (NA) no tiene sótano
#BsmtExposure: Refleja la cantidad de exposición al sótano al aire exterior. -> (NA) no tiene sótano
#BsmtFinType2: Evaluación del tipo de acabado del sótano (si hay dos tipos). -> (NA) no tiene sótano
#BsmtHalfBath: Número de medios baños en el sótano.
#BsmtFullBath: Número de baños completos en el sótano.
cols_not_basement = ['BsmtCond', 'BsmtQual', 'BsmtFinType1', 'BsmtExposure', 'BsmtFinType2']
df_train.loc[cond_not_basement, cols_not_basement] = df_train.loc[cond_not_basement, cols_not_basement].fillna('NA')

cond_no_exposuse = (df_train['TotalBsmtSF'] > 0) & (df_train['BsmtUnfSF'] == 0)


""" cond_unf_basement = (df_train['TotalBsmtSF'] > 0) & (df_train['BsmtUnfSF'] > 0)
#BsmtExposure: Refleja la cantidad de exposición al sótano al aire exterior. -> (NA) no tiene sótano y (No) no tiene exposición
df_train.loc[cond_unf_basement, 'BsmtExposure'] = df_train.loc[cond_unf_basement, 'BsmtExposure'].fillna('No') """




bsmt = df_train[['TotalBsmtSF', 'BsmtUnfSF', 'BsmtCond', 'BsmtQual', 'BsmtHalfBath', 'BsmtFullBath', 'BsmtFinSF1', 'BsmtFinType1', 'BsmtExposure', 'BsmtFinSF2', 'BsmtFinType2']]
bsmt[(bsmt['BsmtFinType1'].isnull()) | (bsmt['BsmtFinType2'].isnull()) | (bsmt['BsmtExposure'].isnull())]

,TotalBsmtSF,BsmtUnfSF,BsmtCond,BsmtQual,BsmtHalfBath,BsmtFullBath,BsmtFinSF1,BsmtFinType1,BsmtExposure,BsmtFinSF2,BsmtFinType2
332,3206,1603,TA,Gd,0,1,1124,GLQ,No,479,NaN
948,936,936,TA,Gd,0,0,0,Unf,NaN,0,Unf


In [ ]:
bsmt.groupby(['BsmtFinSF1', 'BsmtFinType2', 'BsmtExposure']).size()
plt.plot()

BsmtFinSF1  BsmtFinType2  BsmtExposure
0           NA            NA               37
            Unf           Av               46
                          Gd               11
                          Mn               27
                          No              345
                                         ... 
1904        Unf           No                1
2096        Unf           Av                1
2188        Unf           Gd                1
2260        Unf           Gd                1
5644        Unf           Gd                1
Length: 831, dtype: int64